# Rule-Based Strip Centering

This notebook gives us a no-training autonomous step for `02`.

Strategy:

- build the binary lane mask from the live RGB frame
- treat the white strip in that mask as the target
- estimate a near-point and a far-point on the strip
- choose steering direction from the strip error first
- send short drive pulses instead of continuous power
- if the strip does not move closer to center, increase drive slightly on the next pulse

This helps with floor friction because we do not assume one fixed steering angle always produces the same movement.
    


In [ ]:
from pathlib import Path
import glob
import io
import json
import os
import sys
import threading
import time

import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
from IPython.display import display
from PIL import Image
import serial
import serial.tools.list_ports

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from jetcar.camera import open_camera, read_rgb_frame
from jetcar.vision import build_lane_mask

video_devices = sorted(glob.glob('/dev/video*'))
print('Project root:', project_root)
print('Video devices:', video_devices or ['none found'])
print('Serial has Serial:', hasattr(serial, 'Serial'))
    


In [ ]:
camera_handle = None
serial_handle = None
current_frame = None
last_analysis = None
auto_thread = None
auto_stop_event = threading.Event()
controller_state = {
    'drive': 0.18,
    'last_metric': None,
    'steps': 0,
    'stuck_steps': 0,
    'move_streak': 0,
}


def available_ports():
    ports = [port.device for port in serial.tools.list_ports.comports()]
    for fallback in ('/dev/ttyTHS1', '/dev/ttyTHS2'):
        if os.path.exists(fallback) and fallback not in ports:
            ports.append(fallback)
    return ports


def set_status(text):
    status.value = f'<b>Status:</b> {text}'


def log(message):
    with log_output:
        print(message)


def render_image(widget, image):
    if isinstance(image, np.ndarray):
        pil_image = Image.fromarray(image)
    else:
        pil_image = image
    with io.BytesIO() as buf:
        pil_image.save(buf, format='JPEG')
        widget.value = buf.getvalue()


def open_selected_camera():
    global camera_handle
    close_camera()
    camera_handle = open_camera(
        source=camera_source_dropdown.value,
        sensor_id=sensor_id.value,
        device_index=device_index.value,
        width=frame_width.value,
        height=frame_height.value,
        warmup_frames=warmup_frames.value,
    )
    set_status(f'camera open ({camera_source_dropdown.value})')
    try:
        analyze_with_retries(sample_count=1, retries=0)
    except Exception as exc:
        log(f'PREVIEW: {exc}')
        set_status(f'camera open ({camera_source_dropdown.value}); preview pending')


def close_camera():
    global camera_handle
    if camera_handle is not None:
        camera_handle.release()
    camera_handle = None


def grab_frame(sample_count=3, pause_s=0.02):
    global current_frame
    if camera_handle is None:
        raise RuntimeError('Camera is not open.')
    frame = None
    for _ in range(max(sample_count, 1)):
        frame = read_rgb_frame(camera_handle)
        time.sleep(pause_s)
    current_frame = frame
    render_image(raw_preview, current_frame)
    return current_frame


def open_serial_device(port, baudrate, timeout=0.4):
    ser = serial.Serial(port, baudrate=baudrate, timeout=timeout)
    ser.reset_input_buffer()
    ser.reset_output_buffer()
    return ser


def close_serial_device():
    global serial_handle
    if serial_handle is not None and serial_handle.is_open:
        serial_handle.close()
    serial_handle = None


def send_json(payload):
    if serial_handle is None or not serial_handle.is_open:
        raise RuntimeError('Serial port is not open.')
    line = json.dumps(payload, separators=(',', ':')) + '\n'
    serial_handle.write(line.encode('utf-8'))
    serial_handle.flush()
    time.sleep(0.08)
    waiting = serial_handle.in_waiting
    response = ''
    if waiting:
        response = serial_handle.read(waiting).decode('utf-8', errors='replace')
    return line.strip(), response


def stop_now():
    sent, response = send_json({'T': 1, 'L': 0.0, 'R': 0.0})
    log(f'STOP {sent}')
    if response:
        log(f'RECV {response}')


def pulse_drive(left, right, duration_s):
    sent, response = send_json({'T': 1, 'L': float(left), 'R': float(right)})
    log(f'DRIVE {sent}')
    if response:
        log(f'RECV {response}')
    time.sleep(duration_s)
    stop_now()


def on_click_wrap(fn):
    def wrapped(_):
        try:
            fn()
        except Exception as exc:
            set_status(f'error: {exc}')
            log(f'ERROR: {exc}')
    return wrapped


PRESETS = {
    'green_track': {
        'crop_top_ratio': 0.35,
        'near_row_ratio': 0.82,
        'far_row_ratio': 0.42,
        'h_min': 35,
        'h_max': 95,
        's_min': 40,
        's_max': 255,
        'v_min': 40,
        'v_max': 255,
        'blur_kernel': 5,
        'morph_kernel': 5,
    },
    'yellow_on_red': {
        'crop_top_ratio': 0.35,
        'near_row_ratio': 0.82,
        'far_row_ratio': 0.42,
        'h_min': 8,
        'h_max': 55,
        's_min': 20,
        's_max': 255,
        'v_min': 80,
        'v_max': 255,
        'blur_kernel': 5,
        'morph_kernel': 5,
    },
}


def apply_preset(name):
    preset = PRESETS[name]
    crop_top_ratio.value = preset['crop_top_ratio']
    near_row_ratio.value = preset['near_row_ratio']
    far_row_ratio.value = preset['far_row_ratio']
    h_min.value = preset['h_min']
    h_max.value = preset['h_max']
    s_min.value = preset['s_min']
    s_max.value = preset['s_max']
    v_min.value = preset['v_min']
    v_max.value = preset['v_max']
    blur_kernel.value = preset['blur_kernel']
    morph_kernel.value = preset['morph_kernel']
    set_status(f'preset loaded: {name}')
    if camera_handle is not None:
        try:
            analyze_with_retries(sample_count=1, retries=0)
        except Exception as exc:
            log(f'PREVIEW: {exc}')
            set_status(f'preset loaded: {name}; preview pending')



def learn_target_from_click():
    frame = grab_frame(sample_count=2, pause_s=0.03)

    plt.figure(figsize=(10, 6))
    plt.imshow(frame)
    plt.title('Click near the center of the target strip')
    plt.axis('on')
    pts = plt.ginput(1, timeout=-1)
    plt.close()

    if len(pts) != 1:
        raise RuntimeError('Expected 1 click on the target strip.')

    x, y = pts[0]
    x = int(round(x))
    y = int(round(y))
    radius = int(sample_radius.value)

    x0 = max(0, x - radius)
    x1 = min(frame.shape[1], x + radius + 1)
    y0 = max(0, y - radius)
    y1 = min(frame.shape[0], y + radius + 1)
    patch = frame[y0:y1, x0:x1]
    if patch.size == 0:
        raise RuntimeError('Sample patch was empty.')

    hsv = cv2.cvtColor(patch, cv2.COLOR_RGB2HSV)
    h = hsv[:, :, 0].reshape(-1)
    s = hsv[:, :, 1].reshape(-1)
    v = hsv[:, :, 2].reshape(-1)

    h_lo = max(0, int(np.percentile(h, 10)) - int(h_margin.value))
    h_hi = min(179, int(np.percentile(h, 90)) + int(h_margin.value))
    s_lo = max(0, int(np.percentile(s, 10)) - int(s_margin.value))
    s_hi = min(255, int(np.percentile(s, 98)) + int(s_margin.value))
    v_lo = max(0, int(np.percentile(v, 10)) - int(v_margin.value))
    v_hi = min(255, int(np.percentile(v, 98)) + int(v_margin.value))

    h_min.value = h_lo
    h_max.value = h_hi
    s_min.value = s_lo
    s_max.value = s_hi
    v_min.value = v_lo
    v_max.value = v_hi

    log(f'LEARN click=({x},{y}) patch=({x0}:{x1},{y0}:{y1}) hsv=H[{h_lo},{h_hi}] S[{s_lo},{s_hi}] V[{v_lo},{v_hi}]')
    set_status('learned HSV from clicked strip patch')
    analyze_with_retries(sample_count=1, retries=0)



def learn_center_patch():
    frame = grab_frame(sample_count=2, pause_s=0.03)
    height, width = frame.shape[:2]
    radius = int(sample_radius.value)
    x = width // 2
    y = min(height - radius - 1, max(radius, int(height * center_sample_y_ratio.value)))

    x0 = max(0, x - radius)
    x1 = min(width, x + radius + 1)
    y0 = max(0, y - radius)
    y1 = min(height, y + radius + 1)
    patch = frame[y0:y1, x0:x1]
    if patch.size == 0:
        raise RuntimeError('Center sample patch was empty.')

    hsv = cv2.cvtColor(patch, cv2.COLOR_RGB2HSV)
    h = hsv[:, :, 0].reshape(-1)
    s = hsv[:, :, 1].reshape(-1)
    v = hsv[:, :, 2].reshape(-1)

    h_lo = max(0, int(np.percentile(h, 10)) - int(h_margin.value))
    h_hi = min(179, int(np.percentile(h, 90)) + int(h_margin.value))
    s_lo = max(0, int(np.percentile(s, 10)) - int(s_margin.value))
    s_hi = min(255, int(np.percentile(s, 98)) + int(s_margin.value))
    v_lo = max(0, int(np.percentile(v, 10)) - int(v_margin.value))
    v_hi = min(255, int(np.percentile(v, 98)) + int(v_margin.value))

    h_min.value = h_lo
    h_max.value = h_hi
    s_min.value = s_lo
    s_max.value = s_hi
    v_min.value = v_lo
    v_max.value = v_hi

    log(f'LEARN center=({x},{y}) patch=({x0}:{x1},{y0}:{y1}) hsv=H[{h_lo},{h_hi}] S[{s_lo},{s_hi}] V[{v_lo},{v_hi}]')
    set_status('learned HSV from center patch')
    analyze_with_retries(sample_count=1, retries=0)


In [ ]:
def compute_row_center(mask, row_ratio):
    row_index = int(mask.shape[0] * row_ratio)
    row_index = max(0, min(mask.shape[0] - 1, row_index))
    xs = np.flatnonzero(mask[row_index] > 0)
    if xs.size == 0:
        return None, row_index
    return float(xs.mean()), row_index


def analyze_frame(frame_rgb):
    height, width = frame_rgb.shape[:2]
    scale = float(analysis_scale.value)
    if scale < 0.99:
        scaled_width = max(64, int(width * scale))
        scaled_height = max(48, int(height * scale))
        working = cv2.resize(frame_rgb, (scaled_width, scaled_height), interpolation=cv2.INTER_AREA)
    else:
        working = frame_rgb
        scaled_height, scaled_width = height, width

    mask_image = build_lane_mask(
        Image.fromarray(working),
        crop_top_ratio=crop_top_ratio.value,
        hsv_lower=(h_min.value, s_min.value, v_min.value),
        hsv_upper=(h_max.value, s_max.value, v_max.value),
        blur_kernel=blur_kernel.value,
        morph_kernel=morph_kernel.value,
    )
    mask_small = np.array(mask_image)

    x_near_small, row_near_small = compute_row_center(mask_small, near_row_ratio.value)
    x_far_small, row_far_small = compute_row_center(mask_small, far_row_ratio.value)
    if x_near_small is None or x_far_small is None:
        raise RuntimeError('Could not find the target strip in the mask. Adjust HSV or camera framing.')

    x_scale = width / float(scaled_width)
    y_scale = height / float(scaled_height)
    x_near = x_near_small * x_scale
    x_far = x_far_small * x_scale
    row_near = row_near_small * y_scale
    row_far = row_far_small * y_scale

    image_center_x = width / 2.0
    offset_norm = (x_near - image_center_x) / (width / 2.0)
    heading_norm = (x_far - x_near) / (width / 2.0)
    steering = float(np.clip(k_offset.value * offset_norm + k_heading.value * heading_norm, -1.0, 1.0))
    metric = abs(offset_norm) + heading_weight.value * abs(heading_norm)

    overlay = frame_rgb.copy()
    cv2.line(overlay, (int(image_center_x), 0), (int(image_center_x), height - 1), (255, 0, 0), 2)
    cv2.circle(overlay, (int(x_near), int(row_near)), 8, (0, 255, 0), -1)
    cv2.circle(overlay, (int(x_far), int(row_far)), 8, (255, 255, 0), -1)
    cv2.line(overlay, (int(x_near), int(row_near)), (int(x_far), int(row_far)), (0, 255, 0), 3)
    cv2.putText(overlay, f'offset={offset_norm:+.3f}', (20, 36), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(overlay, f'heading={heading_norm:+.3f}', (20, 72), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(overlay, f'steer={steering:+.3f}', (20, 108), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(overlay, f'scale={scale:.2f}', (20, 144), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, cv2.LINE_AA)

    if scaled_width != width or scaled_height != height:
        mask = cv2.resize(mask_small, (width, height), interpolation=cv2.INTER_NEAREST)
    else:
        mask = mask_small
    mask_rgb = np.stack([mask] * 3, axis=-1)
    return {
        'mask_rgb': mask_rgb,
        'overlay': overlay,
        'offset_norm': float(offset_norm),
        'heading_norm': float(heading_norm),
        'steering': steering,
        'metric': float(metric),
        'x_near': float(x_near),
        'x_far': float(x_far),
        'row_near': int(row_near),
        'row_far': int(row_far),
    }


def analyze_current_frame(sample_count=None):
    global last_analysis
    if sample_count is None:
        sample_count = analysis_samples.value
    frame = grab_frame(sample_count=sample_count, pause_s=analysis_sample_pause.value)
    last_analysis = analyze_frame(frame)
    render_image(mask_preview, last_analysis['mask_rgb'])
    render_image(overlay_preview, last_analysis['overlay'])
    metrics.value = (
        f"<b>Offset:</b> {last_analysis['offset_norm']:+.3f} &nbsp; "
        f"<b>Heading:</b> {last_analysis['heading_norm']:+.3f} &nbsp; "
        f"<b>Steering:</b> {last_analysis['steering']:+.3f} &nbsp; "
        f"<b>Metric:</b> {last_analysis['metric']:.3f}"
    )
    set_status('analysis ready')
    return last_analysis


def analyze_with_retries(sample_count=None, retries=None):
    if sample_count is None:
        sample_count = analysis_samples.value
    if retries is None:
        retries = reacquire_retries.value
    last_exc = None
    for attempt in range(retries + 1):
        try:
            return analyze_current_frame(sample_count=sample_count)
        except RuntimeError as exc:
            last_exc = exc
            if 'target strip' not in str(exc):
                raise
            if attempt >= retries:
                raise
            time.sleep(reacquire_wait.value)
    raise last_exc


def compute_drive_from_analysis(analysis):
    steer = float(analysis['steering'])
    drive = float(controller_state['drive'])
    stuck_steps = int(controller_state['stuck_steps'])
    move_streak = int(controller_state.get('move_streak', 0))
    mode = 'normal'

    if abs(steer) < steer_deadband.value:
        steer = 0.0
    elif abs(steer) < min_steer.value:
        steer = np.sign(steer) * min_steer.value

    if abs(analysis['offset_norm']) <= center_tolerance.value and abs(analysis['heading_norm']) <= heading_tolerance.value:
        drive = min(drive, centered_drive.value)

    turn_gain = turn_mix.value
    pulse_duration = pulse_time.value + min(move_streak, max_move_streak.value) * move_hold_step.value
    pulse_duration = min(max_pulse_time.value, pulse_duration)

    if stuck_steps >= stuck_after.value:
        drive = min(max_drive.value, drive + stuck_drive_boost.value)
        turn_gain = min(max_turn_mix.value, turn_gain + stuck_turn_boost.value)
        pulse_duration = min(max_pulse_time.value, pulse_duration + stuck_pulse_boost.value)
        mode = 'boost'

    if stuck_steps >= pivot_after.value:
        mode = 'pivot'
        pivot_sign = 1.0 if steer >= 0 else -1.0
        pivot_drive = min(pivot_drive_value.value, max_drive.value)
        left = float(np.clip(pivot_sign * pivot_drive, -1.0, 1.0))
        right = float(np.clip(-pivot_sign * pivot_drive, -1.0, 1.0))
        pulse_duration = min(max_pulse_time.value, max(pulse_duration, pivot_hold_time.value))
        return pivot_drive, left, right, pulse_duration, pivot_sign, max_turn_mix.value, mode

    steer_term = turn_gain * steer
    left = float(np.clip(drive + steer_term, -1.0, 1.0))
    right = float(np.clip(drive - steer_term, -1.0, 1.0))
    return drive, left, right, pulse_duration, steer, turn_gain, mode


def judge_improvement(before_analysis, after_analysis):
    metric_drop = float(before_analysis['metric'] - after_analysis['metric'])
    offset_drop = float(abs(before_analysis['offset_norm']) - abs(after_analysis['offset_norm']))
    heading_drop = float(abs(before_analysis['heading_norm']) - abs(after_analysis['heading_norm']))
    steer_drop = float(abs(before_analysis['steering']) - abs(after_analysis['steering']))
    x_shift = float(abs(after_analysis['x_near'] - before_analysis['x_near']))

    metric_ok = metric_drop >= improvement_margin.value
    offset_ok = offset_drop >= min_offset_improvement.value
    heading_ok = heading_drop >= min_heading_improvement.value
    steer_ok = steer_drop >= min_steer_improvement.value
    shift_ok = x_shift >= min_strip_shift_px.value

    improved = metric_ok and shift_ok and (offset_ok or heading_ok or steer_ok)
    reason = (
        f'metric={metric_drop:+.3f} offset={offset_drop:+.3f} '
        f'heading={heading_drop:+.3f} steer={steer_drop:+.3f} shift={x_shift:.1f}px'
    )
    return improved, reason, metric_drop, offset_drop, heading_drop, steer_drop, x_shift


def update_drive_level(before_analysis, after_analysis):
    before_metric = float(before_analysis['metric'])
    after_metric = float(after_analysis['metric'])
    improved, reason, metric_drop, offset_drop, heading_drop, steer_drop, x_shift = judge_improvement(before_analysis, after_analysis)

    if improved:
        controller_state['stuck_steps'] = 0
        controller_state['move_streak'] = int(controller_state.get('move_streak', 0)) + 1
        controller_state['drive'] = max(min_drive.value, controller_state['drive'] - relax_step.value)
    else:
        controller_state['move_streak'] = 0
        controller_state['stuck_steps'] += 1
        controller_state['drive'] = min(max_drive.value, controller_state['drive'] + drive_step.value)
        if controller_state['stuck_steps'] >= stuck_after.value:
            controller_state['drive'] = min(max_drive.value, controller_state['drive'] + stuck_drive_boost.value)

    centered_enough = (
        after_metric <= lock_metric.value
        and abs(after_analysis['offset_norm']) <= center_tolerance.value
        and abs(after_analysis['heading_norm']) <= heading_tolerance.value
        and abs(after_analysis['steering']) <= lock_steer_tolerance.value
    )
    if centered_enough and improved:
        controller_state['stuck_steps'] = 0
        controller_state['move_streak'] = min(controller_state['move_streak'], 1)
        controller_state['drive'] = min(controller_state['drive'], centered_drive.value)

    controller_state['last_metric'] = after_metric
    return {
        'improved': improved,
        'before_metric': before_metric,
        'after_metric': after_metric,
        'next_drive': controller_state['drive'],
        'stuck_steps': controller_state['stuck_steps'],
        'move_streak': controller_state['move_streak'],
        'reason': reason,
        'metric_drop': metric_drop,
        'offset_drop': offset_drop,
        'heading_drop': heading_drop,
        'steer_drop': steer_drop,
        'x_shift': x_shift,
    }


def auto_step():
    before = analyze_with_retries(sample_count=analysis_samples.value, retries=0)
    drive, left, right, hold_time, steer_used, turn_gain, mode = compute_drive_from_analysis(before)
    direction = 'right' if steer_used > 0.03 else 'left' if steer_used < -0.03 else 'straight'
    pulse_drive(left, right, hold_time)
    time.sleep(settle_time.value)
    after = analyze_with_retries(sample_count=post_move_samples.value, retries=reacquire_retries.value)
    result = update_drive_level(before, after)
    controller_state['steps'] += 1
    decision.value = (
        f"<b>Mode:</b> {mode} &nbsp; "
        f"<b>Direction:</b> {direction} &nbsp; "
        f"<b>Hold:</b> {hold_time:.2f}s &nbsp; "
        f"<b>Drive:</b> {drive:.2f} &nbsp; "
        f"<b>Turn mix:</b> {turn_gain:.2f} &nbsp; "
        f"<b>Left/Right:</b> {left:.2f} / {right:.2f} &nbsp; "
        f"<b>Improved:</b> {'yes' if result['improved'] else 'no'} &nbsp; "
        f"<b>Stuck:</b> {result['stuck_steps']} &nbsp; "
        f"<b>Move streak:</b> {result['move_streak']}"
    )
    log(
        f"STEP {controller_state['steps']:03d} mode={mode} dir={direction} hold={hold_time:.2f}s metric {result['before_metric']:.3f} -> {result['after_metric']:.3f} next_drive={result['next_drive']:.2f} stuck={result['stuck_steps']} move_streak={result['move_streak']} {result['reason']}"
    )
    set_status('auto step complete')


def auto_loop_worker():
    try:
        while not auto_stop_event.is_set():
            auto_step()
            time.sleep(loop_pause.value)
    except Exception as exc:
        log(f'ERROR: {exc}')
        set_status(f'error: {exc}')
    finally:
        try:
            if serial_handle is not None and serial_handle.is_open:
                stop_now()
        except Exception:
            pass
        auto_stop_event.set()


def start_auto_loop():
    global auto_thread
    if auto_thread is not None and auto_thread.is_alive():
        raise RuntimeError('Auto loop is already running.')
    auto_stop_event.clear()
    auto_thread = threading.Thread(target=auto_loop_worker, daemon=True)
    auto_thread.start()
    set_status('auto loop running')


def stop_auto_loop():
    auto_stop_event.set()
    set_status('auto loop stopping')


def reset_controller():
    controller_state['drive'] = float(min_drive.value)
    controller_state['last_metric'] = None
    controller_state['steps'] = 0
    controller_state['stuck_steps'] = 0
    controller_state['move_streak'] = 0
    decision.value = '<b>Mode:</b> -- <b>Direction:</b> -- <b>Hold:</b> -- <b>Drive:</b> -- <b>Improved:</b> --'
    set_status(f'controller reset at drive={controller_state["drive"]:.2f}')


In [ ]:
port_options = available_ports() or ['/dev/ttyTHS1', '/dev/ttyTHS2']
default_port = '/dev/ttyTHS1' if '/dev/ttyTHS1' in port_options else port_options[0]

camera_source_dropdown = widgets.Dropdown(options=['usb', 'csi'], value='usb', description='Source')
sensor_id = widgets.IntText(value=0, description='Sensor')
device_index = widgets.IntText(value=0, description='Device')
frame_width = widgets.IntText(value=1280, description='Width')
frame_height = widgets.IntText(value=720, description='Height')
warmup_frames = widgets.IntText(value=12, description='Warmup')

port_dropdown = widgets.Dropdown(options=port_options, value=default_port, description='Port')
baud_input = widgets.IntText(value=115200, description='Baud')

crop_top_ratio = widgets.FloatSlider(value=0.35, min=0.0, max=0.8, step=0.05, description='Crop Top')
near_row_ratio = widgets.FloatSlider(value=0.82, min=0.50, max=0.98, step=0.02, description='Near Row')
far_row_ratio = widgets.FloatSlider(value=0.42, min=0.05, max=0.80, step=0.02, description='Far Row')
h_min = widgets.IntSlider(value=35, min=0, max=179, step=1, description='H min')
h_max = widgets.IntSlider(value=95, min=0, max=179, step=1, description='H max')
s_min = widgets.IntSlider(value=40, min=0, max=255, step=1, description='S min')
s_max = widgets.IntSlider(value=255, min=0, max=255, step=1, description='S max')
v_min = widgets.IntSlider(value=40, min=0, max=255, step=1, description='V min')
v_max = widgets.IntSlider(value=255, min=0, max=255, step=1, description='V max')
blur_kernel = widgets.IntSlider(value=5, min=1, max=15, step=2, description='Blur')
morph_kernel = widgets.IntSlider(value=5, min=1, max=15, step=2, description='Morph')
preset_green_button = widgets.Button(description='Green Track')
preset_yellow_button = widgets.Button(description='Yellow On Red', button_style='warning')
learn_color_button = widgets.Button(description='Learn From Click', button_style='info')
learn_center_button = widgets.Button(description='Learn Center Patch', button_style='info')
analysis_scale = widgets.FloatSlider(value=0.45, min=0.25, max=1.00, step=0.05, description='Scale')
sample_radius = widgets.IntSlider(value=14, min=3, max=40, step=1, description='Patch px')
center_sample_y_ratio = widgets.FloatSlider(value=0.72, min=0.40, max=0.95, step=0.01, description='Center Y')
h_margin = widgets.IntSlider(value=8, min=0, max=40, step=1, description='H pad')
s_margin = widgets.IntSlider(value=35, min=0, max=120, step=1, description='S pad')
v_margin = widgets.IntSlider(value=35, min=0, max=120, step=1, description='V pad')
k_offset = widgets.FloatSlider(value=0.90, min=0.0, max=2.0, step=0.05, description='K offset')
k_heading = widgets.FloatSlider(value=0.60, min=0.0, max=2.0, step=0.05, description='K heading')
heading_weight = widgets.FloatSlider(value=0.35, min=0.0, max=1.5, step=0.05, description='Head wt')
center_tolerance = widgets.FloatSlider(value=0.06, min=0.01, max=0.30, step=0.01, description='Ctr tol')
heading_tolerance = widgets.FloatSlider(value=0.10, min=0.01, max=0.40, step=0.01, description='Head tol')
lock_metric = widgets.FloatSlider(value=0.08, min=0.02, max=0.40, step=0.01, description='Lock')
lock_steer_tolerance = widgets.FloatSlider(value=0.05, min=0.01, max=0.30, step=0.01, description='Lock str')
steer_deadband = widgets.FloatSlider(value=0.05, min=0.00, max=0.30, step=0.01, description='Deadband')
min_steer = widgets.FloatSlider(value=0.18, min=0.00, max=0.50, step=0.01, description='Min steer')
min_offset_improvement = widgets.FloatSlider(value=0.015, min=0.000, max=0.100, step=0.005, description='Off imp')
min_heading_improvement = widgets.FloatSlider(value=0.015, min=0.000, max=0.100, step=0.005, description='Head imp')
min_steer_improvement = widgets.FloatSlider(value=0.020, min=0.000, max=0.120, step=0.005, description='Steer imp')
min_strip_shift_px = widgets.FloatSlider(value=18.0, min=0.0, max=120.0, step=1.0, description='Shift px')

min_drive = widgets.FloatSlider(value=0.12, min=0.05, max=0.40, step=0.01, description='Min drv')
max_drive = widgets.FloatSlider(value=0.28, min=0.10, max=0.70, step=0.01, description='Max drv')
centered_drive = widgets.FloatSlider(value=0.14, min=0.05, max=0.40, step=0.01, description='Ctr drv')
drive_step = widgets.FloatSlider(value=0.02, min=0.00, max=0.12, step=0.01, description='Up step')
relax_step = widgets.FloatSlider(value=0.01, min=0.00, max=0.08, step=0.01, description='Down step')
turn_mix = widgets.FloatSlider(value=0.35, min=0.10, max=1.20, step=0.05, description='Turn mix')
max_turn_mix = widgets.FloatSlider(value=0.60, min=0.20, max=1.50, step=0.05, description='Max turn')
pulse_time = widgets.FloatSlider(value=0.25, min=0.05, max=1.20, step=0.01, description='Hold s')
max_pulse_time = widgets.FloatSlider(value=0.45, min=0.10, max=1.50, step=0.01, description='Max hold')
settle_time = widgets.FloatSlider(value=0.45, min=0.02, max=1.20, step=0.01, description='Settle s')
loop_pause = widgets.FloatSlider(value=0.35, min=0.00, max=1.00, step=0.01, description='Decision gap')
reacquire_retries = widgets.IntSlider(value=3, min=0, max=10, step=1, description='Retries')
reacquire_wait = widgets.FloatSlider(value=0.25, min=0.00, max=1.00, step=0.01, description='Retry wait')
improvement_margin = widgets.FloatSlider(value=0.015, min=0.00, max=0.08, step=0.002, description='Metric imp')
analysis_samples = widgets.IntSlider(value=2, min=1, max=8, step=1, description='Pre samp')
post_move_samples = widgets.IntSlider(value=3, min=1, max=10, step=1, description='Post samp')
analysis_sample_pause = widgets.FloatSlider(value=0.03, min=0.00, max=0.10, step=0.01, description='Sample gap')
stuck_after = widgets.IntSlider(value=3, min=1, max=6, step=1, description='Boost @')
stuck_drive_boost = widgets.FloatSlider(value=0.02, min=0.00, max=0.20, step=0.01, description='Drv boost')
stuck_turn_boost = widgets.FloatSlider(value=0.10, min=0.00, max=0.60, step=0.05, description='Turn boost')
stuck_pulse_boost = widgets.FloatSlider(value=0.08, min=0.00, max=0.60, step=0.01, description='Hold boost')
pivot_after = widgets.IntSlider(value=6, min=2, max=10, step=1, description='Pivot @')
pivot_drive_value = widgets.FloatSlider(value=0.24, min=0.10, max=0.80, step=0.01, description='Pivot drv')
pivot_hold_time = widgets.FloatSlider(value=0.40, min=0.10, max=1.50, step=0.01, description='Pivot hold')
move_hold_step = widgets.FloatSlider(value=0.05, min=0.00, max=0.30, step=0.01, description='Move hold')
max_move_streak = widgets.IntSlider(value=3, min=0, max=8, step=1, description='Move max')

open_camera_button = widgets.Button(description='Open Camera', button_style='success')
close_camera_button = widgets.Button(description='Close Camera')
analyze_button = widgets.Button(description='Analyze / Refresh Mask', button_style='info')
open_port_button = widgets.Button(description='Open Port', button_style='success')
close_port_button = widgets.Button(description='Close Port')
reset_button = widgets.Button(description='Reset Control')
auto_step_button = widgets.Button(description='Auto Step', button_style='warning')
start_loop_button = widgets.Button(description='Start Auto', button_style='primary')
stop_loop_button = widgets.Button(description='Stop Auto', button_style='danger')
stop_button = widgets.Button(description='Stop Motors', button_style='danger')

metrics = widgets.HTML(value='<b>Offset:</b> -- <b>Heading:</b> -- <b>Steering:</b> -- <b>Metric:</b> --')
decision = widgets.HTML(value='<b>Mode:</b> -- <b>Direction:</b> -- <b>Hold:</b> -- <b>Drive:</b> -- <b>Improved:</b> -- <b>Move streak:</b> --')
status = widgets.HTML(value='<b>Status:</b> idle')
log_output = widgets.Output(layout={'border': '1px solid #444', 'max_height': '240px', 'overflow_y': 'auto'})
raw_preview = widgets.Image(format='jpeg', width=420, height=236)
mask_preview = widgets.Image(format='jpeg', width=420, height=236)
overlay_preview = widgets.Image(format='jpeg', width=420, height=236)

preset_green_button.on_click(on_click_wrap(lambda: apply_preset('green_track')))
preset_yellow_button.on_click(on_click_wrap(lambda: apply_preset('yellow_on_red')))
learn_color_button.on_click(on_click_wrap(lambda: learn_target_from_click()))
learn_center_button.on_click(on_click_wrap(lambda: learn_center_patch()))
open_camera_button.on_click(on_click_wrap(lambda: open_selected_camera()))
close_camera_button.on_click(on_click_wrap(lambda: [stop_auto_loop(), close_camera(), set_status('camera closed')]))
analyze_button.on_click(on_click_wrap(lambda: analyze_with_retries(sample_count=analysis_samples.value, retries=reacquire_retries.value)))
open_port_button.on_click(on_click_wrap(lambda: [close_serial_device(), globals().__setitem__('serial_handle', open_serial_device(port_dropdown.value, baud_input.value)), set_status(f'serial open on {port_dropdown.value}')]))
close_port_button.on_click(on_click_wrap(lambda: [stop_auto_loop(), close_serial_device(), set_status('serial closed')]))
reset_button.on_click(on_click_wrap(lambda: reset_controller()))
auto_step_button.on_click(on_click_wrap(lambda: auto_step()))
start_loop_button.on_click(on_click_wrap(lambda: start_auto_loop()))
stop_loop_button.on_click(on_click_wrap(lambda: stop_auto_loop()))
stop_button.on_click(on_click_wrap(lambda: [stop_auto_loop(), stop_now(), set_status('stop sent')]))

panel = widgets.VBox([
    widgets.HTML(value='<h3>Camera</h3>'),
    widgets.HTML(value=f"<b>Video devices:</b> {' '.join(video_devices) if video_devices else 'none found'}"),
    widgets.HBox([camera_source_dropdown, sensor_id, device_index]),
    widgets.HBox([frame_width, frame_height, warmup_frames]),
    widgets.HBox([open_camera_button, close_camera_button, analyze_button]),
    widgets.HTML(value='<h3>Serial</h3>'),
    widgets.HTML(value=f"<b>Serial ports:</b> {' '.join(port_options) if port_options else 'none found'}"),
    widgets.HBox([port_dropdown, baud_input]),
    widgets.HBox([open_port_button, close_port_button, stop_button]),
    widgets.HTML(value='<h3>Mask Tuning</h3>'),
    crop_top_ratio,
    widgets.HBox([near_row_ratio, far_row_ratio]),
    widgets.HBox([preset_green_button, preset_yellow_button, learn_color_button, learn_center_button]),
    widgets.HBox([h_min, h_max]),
    widgets.HBox([s_min, s_max]),   
    widgets.HBox([v_min, v_max]),
    widgets.HBox([blur_kernel, morph_kernel, analysis_scale]),
    widgets.HBox([sample_radius, center_sample_y_ratio, h_margin, s_margin, v_margin]),
    widgets.HBox([k_offset, k_heading, heading_weight]),
    widgets.HBox([center_tolerance, heading_tolerance, lock_metric, lock_steer_tolerance]),
    widgets.HBox([steer_deadband, min_steer]),
    widgets.HBox([min_offset_improvement, min_heading_improvement, min_steer_improvement, min_strip_shift_px]),
    widgets.HTML(value='<h3>Adaptive Drive</h3>'),
    widgets.HBox([min_drive, max_drive, centered_drive]),
    widgets.HBox([drive_step, relax_step, turn_mix, max_turn_mix]),
    widgets.HBox([pulse_time, max_pulse_time, settle_time, loop_pause]),
    widgets.HBox([reacquire_retries, reacquire_wait]),
    widgets.HBox([analysis_samples, post_move_samples, analysis_sample_pause]),
    widgets.HBox([stuck_after, stuck_drive_boost, stuck_turn_boost, stuck_pulse_boost]),
    widgets.HBox([pivot_after, pivot_drive_value, pivot_hold_time]),
    widgets.HBox([move_hold_step, max_move_streak]),
    widgets.HBox([reset_button, auto_step_button, start_loop_button, stop_loop_button]),
    metrics,
    decision,
    status,
    widgets.HBox([raw_preview, mask_preview, overlay_preview]),
    log_output,
])

display(panel)
reset_controller()


## Suggested Order

1. Put the rover on a stand first.
2. Open camera and serial.
3. Tune the mask until the strip is clearly white in the mask preview.
4. Run `Analyze` and confirm the blue center line and strip points make sense.
5. Press `Reset Control` so the first move starts gentle.
6. Use `Auto Step` a few times before `Start Auto`.

This does not currently look like a memory problem. It is more likely:

- motion blur right after a hard move
- the strip temporarily leaving the expected crop area
- the controller analyzing too soon after a pivot

What changed to help that:

- `Scale` downsamples the image for faster mask processing
- `Retries` and `Retry wait` let the notebook wait and reacquire the strip instead of failing on one bad frame
- `Settle s` and `Decision gap` are higher so we analyze after motion settles

Good starting values:

- `Scale = 0.45`
- `Pre samp = 2`
- `Post samp = 3`
- `Sample gap = 0.03`
- `Min drv = 0.12`
- `Turn mix = 0.35`
- `Hold s = 0.25`
- `Move hold = 0.05`
- `Settle s = 0.45`
- `Decision gap = 0.35`
- `Retries = 3`
- `Retry wait = 0.25`

If the mask still disappears during pivot, reduce `Pivot drv` even more, raise `Settle s`, or delay pivot by raising `Pivot @`.


Extra note:

- `Stuck` now only resets when the rover is actually centered and the step really counted as improvement
- `Move streak` adds a little extra hold time after real progress, so analysis can stay fast without chopping motion too early


Color presets:

- `Green Track` restores the older green-lane workshop mask
- `Yellow On Red` loads a better starting HSV range for your new yellow-tape-on-red-floor setup

For the new environment, click `Yellow On Red` first, then fine-tune from there if needed.


Mask note:

- the notebook detects a chosen color range in HSV
- the detected pixels are shown as white in the mask preview
- so the target is not literally a white tape; white just means 'matched the color filter'


If presets still miss the tape:

- click `Learn From Click`
- click once on the center of the strip in the camera image
- the notebook will estimate an HSV range from that patch and update the sliders for you

This is more robust when the tape color under your room lighting is different from our guessed preset.


If `Learn From Click` does nothing in JupyterLab:

- use `Learn Center Patch` instead
- it samples a square patch near the center-bottom of the frame
- this works well when the tape is already roughly centered in view
